# 01 – Download and convert

Downloads the Alibaba `cluster-trace-gpu-v2020` archives, verifies their sha256 checksums,
extracts them and converts every table to Parquet.

- Inputs: Aliyun OSS (or the Drive backup in `data/raw` if present)
- Outputs: `data/raw/*.tar.gz` and `data/interim/*.parquet` on Drive
- Runtime: CPU, about 5 minutes. Needs to be run only once.

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Clone or update the project repository
import os
from google.colab import userdata

token = userdata.get('GH_TOKEN')
REPO = '/content/tevzi'

if not os.path.exists(REPO):
    !git clone -q https://{token}@github.com/emirsalihoglu/tevzi.git {REPO}
else:
    !git -C {REPO} pull -q

!git -C {REPO} log --oneline -1

41e0f2a (HEAD -> main, origin/main, origin/HEAD) docs: close stage 0, log setup decisions


In [3]:
# Make the package importable and prepare Drive folders
import sys
from pathlib import Path

sys.path.insert(0, f'{REPO}/src')

DRIVE = Path('/content/drive/MyDrive/Tevzi')
for sub in ['data/raw', 'data/interim', 'data/processed',
            'results/models', 'results/figures', 'results/tables']:
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)

import tevzi
print('tevzi package:', tevzi.__file__)
print('Drive folder:', DRIVE)

tevzi package: /content/tevzi/src/tevzi/__init__.py
Drive folder: /content/drive/MyDrive/Tevzi


## 1. Configuration

In [4]:
import hashlib
import shutil

# Official download location (Aliyun OSS) and column header files (GitHub)
OSS_URL = 'https://aliopentrace.oss-cn-beijing.aliyuncs.com/v2020GPUTraces'
HEADER_URL = ('https://raw.githubusercontent.com/alibaba/clusterdata/'
              'master/cluster-trace-gpu-v2020/data')

# sha256 checksums from the official README
EXPECTED = {
    'pai_group_tag_table': '722fef30b7fb7aa50dabd79155614b5423a9d65cf45a9b26c590d57725423a14',
    'pai_instance_table':  '1bf1e423a7ce3f8d086699801c362fd56a7182abdb234139e5ebbed97995ca06',
    'pai_job_table':       '5aad7f7caac501136d14ed6a48e40546f825d7b0617a3a4f337e2348fe0a6cb0',
    'pai_machine_metric':  '53ad917193d3b1dd0f3055e723148b1f36c2f81789b014ea2930a7875892eef5',
    'pai_machine_spec':    'cc0d38a4045af1b1af8179de8b1b54b1ddd995e6160d6d061a6b1000f1276c2d',
    'pai_sensor_table':    '9a0b82e8bdf3949281e4ba1423d9b4b34847e52799eecb138966de46da69c7a0',
    'pai_task_table':      'cd1d6dc3215d2a8607ccf6b6dd952b5db776df86926c73259fea7c1499ac40e5',
}
TABLES = list(EXPECTED)

# Local working folders (fast, deleted when the runtime ends)
LOCAL = Path('/content/work')
RAW_LOCAL = LOCAL / 'raw'
HEADERS_LOCAL = LOCAL / 'headers'
EXTRACT_LOCAL = LOCAL / 'extracted'
INTERIM_LOCAL = LOCAL / 'interim'
for p in [RAW_LOCAL, HEADERS_LOCAL, EXTRACT_LOCAL, INTERIM_LOCAL]:
    p.mkdir(parents=True, exist_ok=True)

# Persistent folders on Drive
RAW_DRIVE = DRIVE / 'data/raw'
INTERIM_DRIVE = DRIVE / 'data/interim'


def sha256(path, chunk=1 << 20):
    """Return the sha256 hex digest of a file, read in chunks."""
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

## 2. Download and verify

In [5]:
# Get the archives: reuse the Drive backup if present, otherwise download from OSS
for table in TABLES:
    local = RAW_LOCAL / f'{table}.tar.gz'
    backup = RAW_DRIVE / f'{table}.tar.gz'
    if local.exists():
        source = 'already local'
    elif backup.exists():
        shutil.copy(backup, local)
        source = 'copied from Drive'
    else:
        !curl -fsSL --retry 5 -o {local} {OSS_URL}/{table}.tar.gz
        source = 'downloaded from OSS'
    print(f'{table:22s} {source}')

# Verify checksums against the official values
for table, expected in EXPECTED.items():
    ok = sha256(RAW_LOCAL / f'{table}.tar.gz') == expected
    print(f'{table:22s} {"OK" if ok else "FAILED"}')
    assert ok, f'Checksum mismatch: {table}'

pai_group_tag_table    copied from Drive
pai_instance_table     copied from Drive
pai_job_table          copied from Drive
pai_machine_metric     copied from Drive
pai_machine_spec       copied from Drive
pai_sensor_table       copied from Drive
pai_task_table         copied from Drive
pai_group_tag_table    OK
pai_instance_table     OK
pai_job_table          OK
pai_machine_metric     OK
pai_machine_spec       OK
pai_sensor_table       OK
pai_task_table         OK


In [6]:
# Back up the verified archives to Drive (skip if an identical copy exists)
for table in TABLES:
    local = RAW_LOCAL / f'{table}.tar.gz'
    backup = RAW_DRIVE / f'{table}.tar.gz'
    if backup.exists() and backup.stat().st_size == local.stat().st_size:
        print(f'{table:22s} backup exists')
    else:
        shutil.copy(local, backup)
        print(f'{table:22s} backed up')

pai_group_tag_table    backup exists
pai_instance_table     backup exists
pai_job_table          backup exists
pai_machine_metric     backup exists
pai_machine_spec       backup exists
pai_sensor_table       backup exists
pai_task_table         backup exists


## 3. Extract and attach column names

The CSV files have no header row; column names come from the official `.header` files.

In [7]:
# Extract archives and download the matching .header files
for table in TABLES:
    print('extracting:', table)
    !tar -xzf {RAW_LOCAL}/{table}.tar.gz -C {EXTRACT_LOCAL}
    !curl -fsSL -o {HEADERS_LOCAL}/{table}.header {HEADER_URL}/{table}.header

# Check that the header column count matches the CSV column count
COLUMNS = {}
for table in TABLES:
    cols = (HEADERS_LOCAL / f'{table}.header').read_text().strip().split(',')
    with open(EXTRACT_LOCAL / f'{table}.csv') as f:
        n_csv = len(f.readline().rstrip('\n').split(','))
    assert len(cols) == n_csv, f'Column count mismatch: {table}'
    COLUMNS[table] = cols
    print(f'{table:22s} {len(cols):2d} columns OK')

extracting: pai_group_tag_table
extracting: pai_instance_table
extracting: pai_job_table
extracting: pai_machine_metric
extracting: pai_machine_spec
extracting: pai_sensor_table
extracting: pai_task_table
pai_group_tag_table     5 columns OK
pai_instance_table      9 columns OK
pai_job_table           6 columns OK
pai_machine_metric     12 columns OK
pai_machine_spec        5 columns OK
pai_sensor_table       16 columns OK
pai_task_table         10 columns OK


## 4. Convert to Parquet

Tables are streamed in 64 MB blocks to keep memory low. IDs, status and type fields
are stored as strings; times and measurements as float64. Compression: zstd.

In [8]:
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq

# Columns stored as float64; every other column is stored as string
NUMERIC = {
    'pai_group_tag_table': [],
    'pai_instance_table':  ['start_time', 'end_time'],
    'pai_job_table':       ['start_time', 'end_time'],
    'pai_machine_metric':  ['start_time', 'end_time', 'machine_cpu_iowait',
                            'machine_cpu_kernel', 'machine_cpu_usr', 'machine_gpu',
                            'machine_load_1', 'machine_net_receive',
                            'machine_num_worker', 'machine_cpu'],
    'pai_machine_spec':    ['cap_cpu', 'cap_mem', 'cap_gpu'],
    'pai_sensor_table':    ['cpu_usage', 'gpu_wrk_util', 'avg_mem', 'max_mem',
                            'avg_gpu_wrk_mem', 'max_gpu_wrk_mem', 'read', 'write',
                            'read_count', 'write_count'],
    'pai_task_table':      ['inst_num', 'start_time', 'end_time',
                            'plan_cpu', 'plan_mem', 'plan_gpu'],
}

for table in TABLES:
    cols = COLUMNS[table]
    types = {c: (pa.float64() if c in NUMERIC[table] else pa.string()) for c in cols}

    reader = pacsv.open_csv(
        EXTRACT_LOCAL / f'{table}.csv',
        read_options=pacsv.ReadOptions(column_names=cols, block_size=64 << 20),
        convert_options=pacsv.ConvertOptions(column_types=types,
                                             strings_can_be_null=True),
    )
    out = INTERIM_LOCAL / f'{table}.parquet'
    n_rows = 0
    with pq.ParquetWriter(out, reader.schema, compression='zstd') as writer:
        for batch in reader:
            writer.write_batch(batch)
            n_rows += batch.num_rows
    print(f'{table:22s} rows={n_rows:>12,d}  size={out.stat().st_size / 2**20:8.1f} MB')

pai_group_tag_table    rows=   1,055,032  size=    44.3 MB
pai_instance_table     rows=   7,522,002  size=   520.0 MB
pai_job_table          rows=   1,055,501  size=    50.5 MB
pai_machine_metric     rows=   2,009,423  size=   174.6 MB
pai_machine_spec       rows=       1,897  size=     0.0 MB
pai_sensor_table       rows=   3,033,232  size=   321.0 MB
pai_task_table         rows=   1,261,050  size=    26.7 MB


## 5. Save to Drive

In [9]:
# Copy Parquet files to Drive and verify by byte size
for table in TABLES:
    src = INTERIM_LOCAL / f'{table}.parquet'
    dst = INTERIM_DRIVE / f'{table}.parquet'
    shutil.copy(src, dst)
    status = 'MATCH' if src.stat().st_size == dst.stat().st_size else 'MISMATCH'
    print(f'{table:22s} {status}')
    assert status == 'MATCH', f'Copy failed: {table}'

print('Done. Parquet tables are in', INTERIM_DRIVE)

pai_group_tag_table    MATCH
pai_instance_table     MATCH
pai_job_table          MATCH
pai_machine_metric     MATCH
pai_machine_spec       MATCH
pai_sensor_table       MATCH
pai_task_table         MATCH
Done. Parquet tables are in /content/drive/MyDrive/Tevzi/data/interim
